### Mem0

In [ ]:
from mem0 import Memory

m = Memory()

# store a memory from a chat turn (Mem0 can infer “facts/preferences” from messages)
messages = [
    {"role": "user", "content": "I travel a lot for work, prefer aisle seats."},
    {"role": "assistant", "content": "Noted. I’ll prioritize aisle seats in bookings."}
]

m.add(messages, user_id="alice", agent_id="travel-assistant", run_id="sess-001",
        metadata={"category": "travel_prefs"})

# retrieve all memories for a user (or scope by agent_id / run_id)
all_user_memories = m.get_all(user_id="alice")

# semantic search over memories for context building
relevant = m.search(query="What seat does Alice prefer?", user_id="alice")

# update or inspect history for a specific memory
mem_id = relevant[0]["id"]
m.update(memory_id=mem_id, data="User prefers aisle seats and avoids red-eye flights.")
history = m.history(memory_id=mem_id)

# build an LLM prompt with retrieved memory
def build_prompt(user_query: str, memories: list[str]) -> str:
    memtext = "\n".join(f"- {m['memory']}" for m in memories)
    return f"""You are a travel assistant.
               Known user memory:
               {memtext}
               User: {user_query}
               Answer with these memories in mind."""

prompt = build_prompt("Book me to NYC next week", relevant)
print(prompt)

# delete (single / all for user)
m.delete(memory_id=mem_id)
m.delete_all(user_id="alice")